In [ ]:
import OptimusPrimus

from OptimusPrimus import OptimusPrimus

print("Libraries imported successfully.")

In [ ]:
# --- Inference configuration ---

# Must match the sub-folder name under Data/images/<IMAGE_SPEC>/tiles/
IMAGE_SPEC = "ARCI URAS26F2"

# The encoder/model-type names below must match what the models were trained with
# (SEG_MODEL_CONFIG['encoder'] / TRAIN_MODEL_CONFIG['seg_encoder'] and
#  CLASS_MODEL_CONFIG['encoder'] / TRAIN_MODEL_CONFIG['class_model_type']).
# Defaults shown here match the library defaults (SEG_MODEL_CONFIG / CLASS_MODEL_CONFIG).
SEG_ENCODER = "efficientnet-b7"
CLASS_MODEL_TYPE = "efficientnet_b0"

# OptimusPrimusTraining's "strict dynamic naming scheme" saves checkpoints as:
#   Data/models/segmentation/seg_model_{encoder}_{image_spec}.pth
#   Data/models/classification/class_model_{class_model_type}_{image_spec}.pth
# so, assuming the models were trained and saved with this standard naming, we can
# reconstruct their spec strings (without the .pth extension) directly:
SEG_MODEL_SPEC = f"seg_model_{SEG_ENCODER}_{IMAGE_SPEC}"
CLASS_MODEL_SPEC = f"class_model_{CLASS_MODEL_TYPE}_{IMAGE_SPEC}"

# Optional configuration dictionaries (only specified keys override the library defaults).
IMAGE_CONFIG = {}
SEG_MODEL_CONFIG = {}
CLASS_MODEL_CONFIG = {}

print(f"Segmentation model spec  : {SEG_MODEL_SPEC}")
print(f"Classification model spec: {CLASS_MODEL_SPEC}")

In [ ]:
training_run = False

if training_run:

    from OptimusPrimus import OptimusPrimusTraining

    TRAIN_IMAGE_DIR = f"Data/images/{IMAGE_SPEC}"

    TRAIN_IMAGE = {}
    TRAIN_MODEL_CONFIG = {}
    TRAINING_PARAMETERS = {}

    train_track_detector = OptimusPrimusTraining(
        TRAIN_IMAGE_DIR,
        IMAGE_SPEC,
        train_image=TRAIN_IMAGE,
        train_model_config=TRAIN_MODEL_CONFIG,
        training_parameters=TRAINING_PARAMETERS,
        image_config=IMAGE_CONFIG,
        parallel=True,  # no GPU -> spread across all CPU cores; set False for single-core
    )

    train_track_detector.perform_seg_training()

    train_track_detector.perform_class_training(train_track_detector.seg_best_model_path, seg_th=0.1)

    dict_seg_efficiency = train_track_detector.evaluate_binned_efficiency(
        'seg',
        seg_th=0.1,
        cls_th=0.55,
    )

    dict_seg_class_efficiency = train_track_detector.evaluate_binned_efficiency(
        'seg_class',
        seg_th=0.1,
        cls_th=0.55,
    )

    print(dict_seg_efficiency['efficiency_table'])

In [ ]:
# --- Build the inference pipeline ---
# `parallel` follows the standard compute policy: a CUDA GPU is always used when available;
# with no GPU, CPU-bound steps (e.g. image quality analysis) are spread across all CPU cores
# unless parallel=False, in which case they run on a single core.
track_detector = OptimusPrimus(
    IMAGE_SPEC,
    SEG_MODEL_SPEC,
    CLASS_MODEL_SPEC,
    image_config=IMAGE_CONFIG,
    seg_model_config=SEG_MODEL_CONFIG,
    class_model_config=CLASS_MODEL_CONFIG,
    parallel=True,
)

print(f"Image tiles folder        : {track_detector.image_path}")
print(f"Segmentation model file   : {track_detector.seg_model_path}")
print(f"Classification model file : {track_detector.cls_model_path}")
print(f"Compute device            : {track_detector.device}")

In [ ]:
# --- Run the full segmentation + classification inference pipeline ---
# seg_th / cls_th default to the model configs' thresholds (SEG_MODEL_CONFIG /
# CLASS_MODEL_CONFIG) when omitted; passed explicitly here for clarity.
track_detector.perform_full_inference(seg_th=track_detector.seg_model_th, cls_th=track_detector.cls_model_th)

print(f"\nSegmentation-only tracks found      : {len(track_detector.seg_ellipses)}")
print(f"Segmentation + classification tracks: {len(track_detector.seg_cls_ellipses)}")
track_detector.seg_cls_ellipses.head()

In [ ]:
# --- Distribution statistics, straight from the just-computed in-memory results ---
seg_stats = track_detector.get_track_distributions('seg')
seg_cls_stats = track_detector.get_track_distributions('seg_class')

seg_stats, seg_cls_stats

In [ ]:
# --- Reload previously saved results from disk (no re-run needed) ---
# Output CSV paths are built and stored on the object itself, so there's no need to
# hand-assemble the (dynamic) filename.
print(f"Segmentation CSV          : {track_detector.seg_output_path}")
print(f"Segmentation+classification CSV: {track_detector.seg_cls_output_path}")

track_detector.inference_from_file(track_detector.seg_output_path)
track_detector.get_track_distributions('seg')

track_detector.inference_from_file(track_detector.seg_cls_output_path)
track_detector.get_track_distributions('seg_class')